In [1]:
import yfinance as yf
import pandas as pd 
import numpy as np

univers = {
    "Maybank": "1155.KL",
    "CIMB": "1023.KL",
    "Public Bank": "1295.KL",
    "Tenaga Nasional": "5347.KL",
    "Petronas Chemicals": "5183.KL"
}
tickers = [c for c in univers.values()]

In [2]:
end_date = pd.to_datetime("2023-01-01")
start_date = end_date - pd.DateOffset(years=5)

data = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=True,
    threads=True
)

[*********************100%***********************]  5 of 5 completed


Imagine qu’une action fasse un split 2 pour 1 :

avant le split, une action valait 100

après le split, elle vaut 50

Si tu regardes les prix bruts, tu vois une grosse baisse soudaine, même si la valeur de l’entreprise n’a pas réellement changé.

Avec auto_adjust=True, pandas/yfinance réécrit les prix pour que cette baisse soit “corrigée” :

le prix avant split devient 100

le prix après split devient 50, mais le cours est ajusté pour garder une série continue
ainsi, l’historique reste plus cohérent pour l’analyse

Exemple concret :

Sans ajustement :

2024-01-01 : 100

2024-01-02 : 50

Avec auto_adjust=True :

le logiciel “recalcule” la série pour que l’évolution soit plus logique et plus comparable dans le temps

Pourquoi c’est utile :

éviter les fausses baisses dues aux splits

avoir une courbe de prix plus stable pour faire des graphiques ou des analyses
En bref :

auto_adjust=True = “corrige les prix pour tenir compte des événements comme les splits”

In [3]:
data = data.stack()
data.columns.name = None
data.reset_index(inplace=True)
data.columns = [c.lower().strip() for c in data.columns]
data.head()


C:\Users\bilal\AppData\Local\Temp\ipykernel_21900\5941902.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  data = data.stack()


,date,ticker,close,high,low,open,volume
0,2018-01-02,1023.KL,4.290183,4.329785,4.270382,4.323185,8969000.0
1,2018-01-02,1155.KL,5.563982,5.575501,5.523663,5.558222,5284100.0
2,2018-01-02,1295.KL,2.995964,3.001737,2.975760,2.995964,10640000.0
3,2018-01-02,5183.KL,5.861957,5.869424,5.735010,5.764880,6016000.0
4,2018-01-02,5347.KL,10.019612,10.098610,9.953780,10.019612,4794000.0


## Définition

La méthode `stack()` **déplace un niveau des colonnes vers l'index**.

- Si les colonnes sont simples (`Index`), elles deviennent un **deuxième niveau de l'index**.
- Si les colonnes sont un `MultiIndex`, `stack()` déplace **le dernier niveau** des colonnes vers l'index (par défaut).

---

## Syntaxe

```python
df.stack(level=-1)
```

- `level=-1` (par défaut) : empile le dernier niveau des colonnes.
- `level=0` ou `level='nom_du_niveau'` : empile un niveau spécifique.

---

In [4]:
data.info()
display(data.describe())
print(data.isnull().sum())
data.drop_duplicates(subset=["date", "ticker"],inplace=True)
data.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6123 entries, 0 to 6122
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    6123 non-null   datetime64[ns]
 1   ticker  6123 non-null   object        
 2   close   6123 non-null   float64       
 3   high    6123 non-null   float64       
 4   low     6123 non-null   float64       
 5   open    6123 non-null   float64       
 6   volume  6123 non-null   float64       
dtypes: datetime64[ns](1), float64(5), object(1)
memory usage: 335.0+ KB


,date,close,high,low,open,volume
count,6123,6123.000000,6123.000000,6123.000000,6123.000000,6.123000e+03
mean,2020-07-01 12:48:19.755021824,5.617246,5.664943,5.570625,5.617563,1.127755e+07
min,2018-01-02 00:00:00,1.983261,2.067855,1.970729,1.998927,0.000000e+00
25%,2019-04-04 00:00:00,3.680041,3.703194,3.653877,3.679455,4.696450e+06
50%,2020-07-02 00:00:00,5.628504,5.672772,5.596384,5.630923,7.939200e+06
75%,2021-09-30 00:00:00,7.162740,7.241148,7.111513,7.176096,1.418790e+07
max,2022-12-30 00:00:00,10.730623,10.757317,10.625264,10.743970,2.236875e+08
std,NaN,2.118302,2.135733,2.102929,2.119348,1.143169e+07


date      0
ticker    0
close     0
high      0
low       0
open      0
volume    0
dtype: int64


,date,ticker,close,high,low,open,volume
0,2018-01-02,1023.KL,4.290183,4.329785,4.270382,4.323185,8969000.0
1,2018-01-02,1155.KL,5.563982,5.575501,5.523663,5.558222,5284100.0
2,2018-01-02,1295.KL,2.995964,3.001737,2.975760,2.995964,10640000.0
3,2018-01-02,5183.KL,5.861957,5.869424,5.735010,5.764880,6016000.0
4,2018-01-02,5347.KL,10.019612,10.098610,9.953780,10.019612,4794000.0
5,2018-01-03,1023.KL,4.309984,4.342985,4.290183,4.316584,24822600.0
6,2018-01-03,1155.KL,5.633100,5.633100,5.563982,5.563982,13946300.0
7,2018-01-03,1295.KL,2.995964,3.010396,2.993078,3.001737,19802000.0
8,2018-01-03,5183.KL,5.936631,6.003838,5.861957,5.861957,7990700.0
9,2018-01-03,5347.KL,10.072278,10.151276,10.019613,10.019613,10311700.0


In [5]:
company_to_ticker = {v: k for k, v in univers.items()}
data["company"] = data["ticker"].map(company_to_ticker)
data = data[["date",
        "company",
        "ticker",
        "open",
        "high",
        "low",
        "close",
        "volume"]]

In [6]:
data = data.sort_values(["ticker","date"]).reset_index(drop=True)
data["rendement_journalier"] = data.groupby("ticker")["close"].pct_change(fill_method=None) * 100
data.set_index(["date","ticker"],inplace=True)
data.head(10)

,,company,open,high,low,close,volume,rendement_journalier
date,ticker,,,,,,,
2018-01-02,1023.KL,CIMB,4.323185,4.329785,4.270382,4.290183,8969000.0,NaN
2018-01-03,1023.KL,CIMB,4.316584,4.342985,4.290183,4.309984,24822600.0,0.461545
2018-01-04,1023.KL,CIMB,4.316584,4.329784,4.296783,4.323184,30095900.0,0.306261
2018-01-05,1023.KL,CIMB,4.329784,4.422188,4.329784,4.422188,35194800.0,2.290078
2018-01-08,1023.KL,CIMB,4.422188,4.507992,4.375987,4.501392,29909700.0,1.791050
2018-01-09,1023.KL,CIMB,4.514592,4.521192,4.428789,4.455190,19028600.0,-1.026398
2018-01-10,1023.KL,CIMB,4.461791,4.474992,4.435390,4.448590,17519400.0,-0.148129
2018-01-11,1023.KL,CIMB,4.422188,4.474991,4.422188,4.455190,16372700.0,0.148349
2018-01-12,1023.KL,CIMB,4.488192,4.514593,4.448590,4.474992,16607900.0,0.444473


In [7]:
data = data.reset_index()
data["variation_intrajournaliere"] = (
    (data["close"] - data["open"])
    / data["open"]
    * 100
)

data["amplitude_journaliere"] = (
    (data["high"] - data["low"])
    / data["open"]
    * 100
)

data["variation_volume"] = (
    data.groupby("ticker")["volume"]
    .pct_change(fill_method=None)
    * 100
)

data["direction"] = np.select(
    [
        data["rendement_journalier"] > 0,
        data["rendement_journalier"] < 0
    ],
    [
        1,
        -1
    ],
    default=0
)
data.head(10)

,date,ticker,company,open,high,low,close,volume,rendement_journalier,variation_intrajournaliere,amplitude_journaliere,variation_volume,direction
0,2018-01-02,1023.KL,CIMB,4.323185,4.329785,4.270382,4.290183,8969000.0,NaN,-0.763362,1.374048,NaN,0
1,2018-01-03,1023.KL,CIMB,4.316584,4.342985,4.290183,4.309984,24822600.0,0.461545,-0.152901,1.223240,176.759951,1
2,2018-01-04,1023.KL,CIMB,4.316584,4.329784,4.296783,4.323184,30095900.0,0.306261,0.152909,0.764522,21.243947,1
3,2018-01-05,1023.KL,CIMB,4.329784,4.422188,4.329784,4.422188,35194800.0,2.290078,2.134144,2.134144,16.942175,1
4,2018-01-08,1023.KL,CIMB,4.422188,4.507992,4.375987,4.501392,29909700.0,1.791050,1.791050,2.985072,-15.016707,1
5,2018-01-09,1023.KL,CIMB,4.514592,4.521192,4.428789,4.455190,19028600.0,-1.026398,-1.315792,2.046782,-36.379837,-1
6,2018-01-10,1023.KL,CIMB,4.461791,4.474992,4.435390,4.448590,17519400.0,-0.148129,-0.295865,0.887580,-7.931219,-1
7,2018-01-11,1023.KL,CIMB,4.422188,4.474991,4.422188,4.455190,16372700.0,0.148349,0.746272,1.194036,-6.545315,1
8,2018-01-12,1023.KL,CIMB,4.488192,4.514593,4.448590,4.474992,16607900.0,0.444473,-0.294117,1.470594,1.436538,1
9,2018-01-15,1023.KL,CIMB,4.494792,4.501392,4.468391,4.481591,15789200.0,0.147473,-0.293685,0.734217,-4.929582,1


In [8]:
from pathlib import Path
output_dir = Path("../data/preprocessed_data")
output_dir.mkdir(parents=True, exist_ok=True)
data.to_csv(output_dir / "stock_data.csv", index=False)

In [9]:
import requests
from bs4 import BeautifulSoup
url = (
    "https://theedgemalaysia.com/"
    "flash-categories/Companies%20in%20the%20news"
)

API_URL = "https://theedgemalaysia.com/api/loadMoreFlashCategories"

headers = {
    "User-Agent": ("Mozilla/5.0")
}

response = requests.get(
    url,
    headers=headers,
    timeout=20
)

print(response.status_code)
print(response.url)
print(len(response.text))

200
https://theedgemalaysia.com/flash-categories/Companies%20in%20the%20news
153364


In [10]:
print(response.text[:500])

<!DOCTYPE html><html><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width"/><title> Companies In The News | The Edge Malaysia</title><script async="" src="https://www.googletagservices.com/tag/js/gpt.js"></script><script>var googletag = window.googletag || {};
        googletag.cmd = googletag.cmd || [];</script><link rel="stylesheet" href="https://myassets.theedgemalaysia.com/css/listing-scriptdms.css "/><script async="" src="https://myassets.theedgemalaysia.com/js/lis


In [11]:
soup = BeautifulSoup(
    response.text,
    "html.parser"
)

print(soup.title)

<title> Companies In The News | The Edge Malaysia</title>


In [12]:
with open("page.html", "w", encoding="utf-8") as f:
    f.write(soup.prettify())


In [13]:
import json

next_data = soup.find(
    "script",
    id="__NEXT_DATA__"
)

data = json.loads(next_data.string)

with open("data.txt", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)


In [14]:
page_props = data["props"]["pageProps"]

initial_articles = page_props["corporateData"]

total_records = page_props["totalRecords"]


In [15]:
import time

all_articles = initial_articles.copy()


# Les 14 premiers sont déjà récupérés
offset = len(initial_articles)


# --------------------------------------------------
# 6. Boucle principale de pagination
# --------------------------------------------------

while offset < total_records:

    params = {
        "offset": offset,
        "flash": "Companies in the news"
    }

    response = requests.get(
        API_URL,
        params=params
    )

    response.raise_for_status()

    data = response.json()

    articles = data["results"]


    # Si l'API ne renvoie plus rien, on arrête
    if not articles:
        print("No more articles returned.")
        break


    # Ajouter les nouveaux articles
    all_articles.extend(articles)


    print(
        f"Offset {offset} -> "
        f"{len(articles)} articles received | "
        f"Total collected: {len(all_articles)}/{total_records}"
    )


    # Avancer selon le nombre réellement reçu
    offset += len(articles)


    # Petite pause pour ne pas surcharger le serveur
    time.sleep(0.2)


# --------------------------------------------------
# 7. Résultat final
# --------------------------------------------------

print("\nFinished.")
print("Expected:", total_records)
print("Collected:", len(all_articles))

Offset 14 -> 10 articles received | Total collected: 24/3465
Offset 24 -> 10 articles received | Total collected: 34/3465
Offset 34 -> 10 articles received | Total collected: 44/3465
Offset 44 -> 10 articles received | Total collected: 54/3465
Offset 54 -> 10 articles received | Total collected: 64/3465
Offset 64 -> 10 articles received | Total collected: 74/3465
Offset 74 -> 10 articles received | Total collected: 84/3465
Offset 84 -> 10 articles received | Total collected: 94/3465
Offset 94 -> 10 articles received | Total collected: 104/3465
Offset 104 -> 10 articles received | Total collected: 114/3465
Offset 114 -> 10 articles received | Total collected: 124/3465
Offset 124 -> 10 articles received | Total collected: 134/3465
Offset 134 -> 10 articles received | Total collected: 144/3465
Offset 144 -> 10 articles received | Total collected: 154/3465
Offset 154 -> 10 articles received | Total collected: 164/3465
Offset 164 -> 10 articles received | Total collected: 174/3465
Offset 17

In [16]:
unique_articles = {}

for article in all_articles:
    unique_articles[article["nid"]] = article

all_articles = list(
    unique_articles.values()
)

print(
    "Unique articles:",
    len(all_articles)
)

Unique articles: 3465


In [17]:
print(all_articles[0])

{'nid': 816945, 'type': 'article', 'language': 'english', 'category': 'Corporate,Malaysia', 'options': 'Top Stories', 'flash': 'Companies in the news', 'tags': '', 'edited': 'S Kanagaraju', 'title': 'Zetrix AI, HeiTech Padu, BCM Alliance, SCIB, Key Asic, HHRG, Shin Yang Group, Khee San & Aumas', 'created': 1788532989000, 'updated': 1788532989000, 'author': 'Jazlin Zakri', 'source': 'theedgemalaysia.com', 'audio': '', 'audioflag': 0, 'alias': 'node/816945', 'video_url': '', 'img': 'https://assets.theedgemarkets.com/Companies-in-the-news-theedgemalaysia-20241211175004-theedgemalaysia_20241211230602_theedgemalaysia_144.jpg', 'caption': '', 'summary': 'Here is a brief recap of some corporate announcements that made the news on Friday.'}


In [18]:
useful_articles = []
for article in all_articles:
    title = article.get("title", "")
    matched_companies = []
    for company in univers.keys():
        if company.lower() in title.lower():
            matched_companies.append(company)
    if matched_companies:
        article["matched_companies"] = matched_companies
        useful_articles.append(article)

print(len(useful_articles))

501


In [20]:
all_company_blocks = []

URL = "https://theedgemalaysia.com/"

for i, article in enumerate(useful_articles):

    print(
        f"\nArticle {i + 1}/{len(useful_articles)}"
    )

    article_url = URL + article["alias"]

    response = requests.get(
        article_url,
        headers=headers,
        timeout=20
    )

    response.raise_for_status()

    article_soup = BeautifulSoup(
        response.text,
        "lxml"
    )

    infos = article_soup.find_all(
        "div",
        class_="newsTextDataWrapInner"
    )


    current_company = None
    current_text = ""


    for info in infos:

        paragraph = info.find("p")

        if not paragraph:
            continue

        bold = paragraph.find("b")


        # Début d'une nouvelle entreprise
        if bold:

            # Sauvegarder le bloc précédent
            if current_company is not None:

                all_company_blocks.append({
                    "nid": article["nid"],
                    "created": article["created"],
                    "article_url": article_url,
                    "article_title": article["title"],
                    "matched_companies": article["matched_companies"],
                    "company": current_company,
                    "text": current_text
                })


            current_company = bold.get_text(
                " ",
                strip=True
            )

            current_text = paragraph.get_text(
                " ",
                strip=True
            )


        # Suite du texte de la même entreprise
        elif current_company is not None:

            current_text += " " + paragraph.get_text(
                " ",
                strip=True
            )


    # Sauvegarder la dernière entreprise de l'article
    if current_company is not None:

        all_company_blocks.append({
            "nid": article["nid"],
            "created": article["created"],
            "article_url": article_url,
            "article_title": article["title"],
            "matched_companies": article["matched_companies"],
            "company": current_company,
            "text": current_text
        })


    time.sleep(0.2)


Article 1/501

Article 2/501

Article 3/501

Article 4/501

Article 5/501

Article 6/501

Article 7/501

Article 8/501

Article 9/501

Article 10/501

Article 11/501

Article 12/501

Article 13/501

Article 14/501

Article 15/501

Article 16/501

Article 17/501

Article 18/501

Article 19/501

Article 20/501

Article 21/501

Article 22/501

Article 23/501

Article 24/501

Article 25/501

Article 26/501

Article 27/501

Article 28/501

Article 29/501

Article 30/501

Article 31/501

Article 32/501

Article 33/501

Article 34/501

Article 35/501

Article 36/501

Article 37/501

Article 38/501

Article 39/501

Article 40/501

Article 41/501

Article 42/501

Article 43/501

Article 44/501

Article 45/501

Article 46/501

Article 47/501

Article 48/501

Article 49/501

Article 50/501

Article 51/501

Article 52/501

Article 53/501

Article 54/501

Article 55/501

Article 56/501

Article 57/501

Article 58/501

Article 59/501

Article 60/501

Article 61/501

Article 62/501

Article 63/501



In [21]:
useful_company_blocks = []

for company in univers.keys():

    company_blocks = [
        block for block in all_company_blocks
        if company.lower() in block["company"].lower()
    ]

    useful_company_blocks.extend(company_blocks)

    print(
        f"{company}: {len(company_blocks)} blocks"
    )

print(
    "Total useful blocks:",
    len(useful_company_blocks)
)

Maybank: 1 blocks
CIMB: 26 blocks
Public Bank: 17 blocks
Tenaga Nasional: 6 blocks
Petronas Chemicals: 8 blocks
Total useful blocks: 58


In [22]:
dataset_rows = []

for company, ticker in univers.items():

    for block in all_company_blocks:

        if company.lower() in block["company"].lower():

            dataset_rows.append({
                "company": company,
                "ticker": ticker,
                "detected_name": block["company"],
                "created": block["created"],
                "text": block["text"],
                "nid": block["nid"],
                "article_title": block["article_title"],
                "article_url": block["article_url"]
            })

df_news = pd.DataFrame(dataset_rows)
df_news["date"] = pd.to_datetime(df_news["created"],unit="ms")
df_news = df_news.drop(columns=["created"])
df_news.head()


,company,ticker,detected_name,text,nid,article_title,article_url,date
0,Maybank,1155.KL,Malayan Banking Bhd (Maybank),Malayan Banking Bhd (Maybank) has dismissed al...,604978,"Luster Industries, Aimflex, Opcom, TM, Maybank...",https://theedgemalaysia.com/article/luster-ind...,2022-01-24 14:32:04
1,CIMB,1023.KL,CIMB Group Holdings Bhd,CIMB Group Holdings Bhd ’s (KL: CIMB ) net pro...,816234,"Yinson, Gamuda, WCT, Zetrix AI, HeiTech Padu, ...",https://theedgemalaysia.com/node/816234,2026-08-28 15:49:34
2,CIMB,1023.KL,CIMB Group Holdings Bhd,CIMB Group Holdings Bhd (KL: CIMB ) posted a s...,805091,"Berjaya, Hong Leong Bank, UOA Development, Hon...",https://theedgemalaysia.com/node/805091,2026-05-26 15:54:50
3,CIMB,1023.KL,CIMB Group Holdings Bhd,CIMB Group Holdings Bhd (KL: CIMB ) has appoin...,802771,"SD Guthrie, Sime Darby, CIMB, AirAsia X, Lotte...",https://theedgemalaysia.com/node/802771,2026-05-07 14:57:58
4,CIMB,1023.KL,CIMB Group Holdings Bhd,CIMB Group Holdings Bhd (KL: CIMB ) is selling...,802431,"Public Bank, CIMB Group, Gamuda, NexG, Telekom...",https://theedgemalaysia.com/node/802431,2026-05-05 15:26:43


In [23]:
df_news.to_csv(output_dir / "news_data.csv", index=False)